# 第48章 统计折线图（lineplot）

用lineplot对重复观察进行统计聚合并显示时间趋势和误差。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。

## 数据结构

长表，一列有序X、一列数值Y，可增加分组列。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 errorbar=None 改为 errorbar="sd" 或 errorbar=("ci", 95)，观察误差区间的显示
2. 修改 estimator 为 "median"，对比均值线与中位数线的趋势差异
3. 添加 markers=False 参数，说明标记点对时间序列可读性的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="notebook")
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category = diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value = diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel = taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend = taxis["tip"], sales=taxis["total"],
    conversion = (taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date = pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region = "AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(data=daily, x="date", y="sales", errorbar=None, color="#1a73e8", marker="o", ax=ax)
ax.set(title="每日平均销售额", xlabel="日期", ylabel="销售额")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=daily, x="date", y="sales", hue="region", marker="o", errorbar=None, palette="colorblind", ax=ax)
ax.set(title="区域每日销售趋势", xlabel="日期", ylabel="销售额")
ax.legend(title="区域", frameon=False)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 3. 参数说明

- estimator：聚合函数
- errorbar：误差
- units：个体线
- sort：排序


## 4. 结果解读

默认线是各X位置的均值，阴影是误差区间；先确认聚合口径。


## 常见误区

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
weekly = daily.copy()
weekly["day"] = weekly["date"].dt.day
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(data=weekly, x="day", y="sales", hue="region", style="region", markers=True, dashes=False, errorbar=None, ax=ax)
ax.set(title="按日序号比较区域趋势", xlabel="日", ylabel="销售额")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

用lineplot对重复观察进行统计聚合并显示时间趋势和误差。


### 你已经掌握

- 判断统计折线图（lineplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。 |
| 数据结构 | 长表，一列有序X、一列数值Y，可增加分组列。 |
| 结果解读 | 默认线是各X位置的均值，阴影是误差区间；先确认聚合口径。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 聚合函数 |
| `errorbar` | 误差 |
| `units` | 个体线 |
| `sort` | 排序 |


### 需要注意

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


### 完成检查

- [ ] 能判断什么问题适合使用统计折线图（lineplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
